In [33]:
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans


In [34]:
data_dir = Path("data")
INDIR = Path("data/data_processed")
OUTDIR = Path("data/data_model")

OUTDIR.mkdir(parents=True, exist_ok=True)

In [35]:
arquivo_df = INDIR / "ANALISE_NOTAS_ENEM_MUNICIPIOS_SP_TRATADO.csv"
arquivo_scaled = INDIR / "ANALISE_NOTAS_ENEM_MUNICIPIOS_SP_MODELO.csv"

df_clustering = pd.read_csv(arquivo_df, encoding='utf-8')
X_scaled = pd.read_csv(arquivo_scaled, encoding='utf-8')

In [36]:
kmeans = KMeans(n_clusters=2, n_init=1000, random_state=0)
kmeans.fit(X_scaled)

labels = kmeans.labels_
df_clustering['CLUSTER'] = labels

In [37]:
caminho_cluster = OUTDIR / "ANALISE_NOTAS_ENEM_MUNICIPIOS_SP_CLUSTERS.csv"
df_clustering.to_csv(caminho_cluster, index=False)

In [38]:
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=X_scaled.columns
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

display(centroids.round(2))

,NU_NOTA_CN_MEDIA,NU_NOTA_CH_MEDIA,NU_NOTA_LC_MEDIA,NU_NOTA_MT_MEDIA,NU_NOTA_REDACAO_MEDIA,RENDA_FAMILIAR_SM_LOG_MEDIANA,REGIAO_CAPITAL,REGIAO_INTERIOR,REGIAO_LITORAL
0,-0.85,-0.81,-0.74,-0.86,-0.75,-0.69,0.00,0.87,0.13
1,0.74,0.71,0.65,0.75,0.66,0.60,0.01,0.98,0.01


In [39]:
df_clustering.head()

,NO_MUNICIPIO_PROVA,NU_NOTA_CN_MEDIA,NU_NOTA_CH_MEDIA,NU_NOTA_LC_MEDIA,NU_NOTA_MT_MEDIA,NU_NOTA_REDACAO_MEDIA,RENDA_FAMILIAR_SM_MEDIA,REGIAO,CLUSTER
0,ADAMANTINA,525.810123,554.820000,557.684444,583.616296,695.506173,2.752041,INTERIOR,1
1,AGUDOS,520.218269,540.266346,551.470513,568.333333,689.871795,2.883191,INTERIOR,1
2,AMERICANA,533.845202,558.039175,560.798216,590.087470,704.480571,3.621081,INTERIOR,1
3,AMPARO,531.318845,552.114015,559.059280,588.289583,686.837121,2.927684,INTERIOR,1
4,ANDRADINA,502.544260,524.655960,541.908609,549.461258,665.651214,2.593284,INTERIOR,0


In [40]:
def moda_ou_na(serie):
    moda = serie.mode()
    return moda.iat[0] if not moda.empty else pd.NA

def top2_ou_na(serie):
    if serie.empty:
        return pd.NA
    return " / ".join(serie.value_counts().head(2).index.astype(str))

resumo_clusters = (
    df_clustering
    .groupby("CLUSTER")
    .agg(
        NU_NOTA_CN_MEDIA=("NU_NOTA_CN_MEDIA", "mean"),
        NU_NOTA_CH_MEDIA=("NU_NOTA_CH_MEDIA", "mean"),
        NU_NOTA_LC_MEDIA=("NU_NOTA_LC_MEDIA", "mean"),
        NU_NOTA_MT_MEDIA=("NU_NOTA_MT_MEDIA", "mean"),
        NU_NOTA_REDACAO_MEDIA=("NU_NOTA_REDACAO_MEDIA", "mean"),
        RENDA_FAMILIAR_SM_MEDIA=("RENDA_FAMILIAR_SM_MEDIA", "mean"),
    )
    .reset_index()
    .sort_values("CLUSTER")
    .reset_index(drop=True)
)



resumo_clusters.columns = [col.upper() for col in resumo_clusters.columns]
display(resumo_clusters.round(2))

,CLUSTER,NU_NOTA_CN_MEDIA,NU_NOTA_CH_MEDIA,NU_NOTA_LC_MEDIA,NU_NOTA_MT_MEDIA,NU_NOTA_REDACAO_MEDIA,RENDA_FAMILIAR_SM_MEDIA
0,0,498.13,526.76,542.68,535.31,647.75,2.36
1,1,521.62,546.10,554.40,571.60,687.93,2.95
